In [1]:
import networkx as nx
import polars as pl
import matplotlib.pyplot as plt

In [ ]:
# 1) Load the dataset from the specific year in Parquet format: 
path = "../data/processed/trade_2024.parquet"
df = pl.read_parquet(path)
df

Year,Exporter,Importer,Product_Code,Value_Thousands_USD
u16,cat,cat,u32,f64
2024,"""AFG""","""AGO""",80810,0.176
2024,"""AFG""","""AGO""",330499,2.295
2024,"""AFG""","""AGO""",732510,5.617
2024,"""AFG""","""AGO""",848330,2.42
2024,"""AFG""","""AGO""",853610,0.605
…,…,…,…,…
2024,"""ZMB""","""URY""",610429,0.585
2024,"""ZMB""","""URY""",848490,0.048
2024,"""ZMB""","""URY""",870810,0.02


In [3]:
# 2) Group by country pairs to add up the total value of trade and prevent Networkx from overlapping each edge: 
df = df.group_by(["Exporter", "Importer"]).agg(
    pl.col("Value_Thousands_USD").sum().alias("weight")
)
df

Exporter,Importer,weight
cat,cat,f64
"""FLK""","""GTM""",0.075
"""FJI""","""CXR""",448.59
"""FJI""","""LBR""",8.535
"""ZAF""","""SYR""",717.137
"""ZWE""","""JOR""",15902.136
…,…,…
"""UKR""","""AND""",38.685
"""UKR""","""EST""",94298.852
"""UKR""","""LVA""",313617.714


In [4]:
# 3) Create the graph as a directed graph (Exporter -> Importer) and the edges weighted by its trade value:
graph = nx.from_pandas_edgelist(
    df.to_pandas(),
    source="Exporter",
    target="Importer",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

In [5]:
# 5) Use the Gephi software to visualize in local the graph: 
nx.write_gexf(graph, "world_trade_network.gexf")